# Sanity 7: Intervention Purity Checks
Checks that each baseline changes only the intended variable (weights vs inputs).

In [ ]:
import hashlib
import torch
from homeomorphism import models
from homeomorphism.interventions import load_model_for_baseline, build_prepared_input

def model_hash(m):
    h = hashlib.sha256()
    for k, v in sorted(m.model.state_dict().items()):
        h.update(k.encode('utf-8'))
        # Ensure tensor is contiguous before getting bytes to avoid hardware-dependent strides
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
base = models.load_model('gpt2', weights='trained', seed=0, device=device)
h_base = model_hash(base)
text = 'The quick brown fox jumps over the lazy dog.'

# Get base prepared input to check for mutations
base_prep = build_prepared_input(m=base, text=text, max_tokens=8, baseline='trained', root_seed=0, sample_index=0)
base_input_ids = base_prep.forward_kwargs['input_ids']

for b in ['topological_initialisation', 'maximum_entropy_injection', 'syntactic_disintegration', 'semantic_scrambling']:
    # Note: load_model_for_baseline reloads the model from disk/cache. 
    # For tiny models like gpt2 this is acceptable for a test script.
    mm = load_model_for_baseline(model_name='gpt2', weights='trained', baseline=b, seed=0, device=device)
    h_new = model_hash(mm)
    prep = build_prepared_input(m=mm, text=text, max_tokens=8, baseline=b, root_seed=0, sample_index=0)
    
    print('---', b)
    print('weights changed:', h_new != h_base)
    
    has_input_ids = 'input_ids' in prep.forward_kwargs
    has_inputs_embeds = 'inputs_embeds' in prep.forward_kwargs
    print('has input_ids:', has_input_ids)
    print('has inputs_embeds:', has_inputs_embeds)
    
    if has_input_ids:
        # Check if the inputs actually mutated compared to the base
        inputs_changed = not torch.equal(prep.forward_kwargs['input_ids'], base_input_ids)
        print('input_ids mutated vs base:', inputs_changed)
        
    print('mask shape:', tuple(prep.forward_kwargs['attention_mask'].shape))
    print()
